# Projeto Forecast — 05. Crescimento populacional e idosos

A base `world_population.csv` só tem a população total, então ela não permite falar de faixas etárias. Aqui trazemos do **Banco Mundial** dois indicadores de idosos (65 anos ou mais) e cruzamos com a nossa base para responder:

1. Quais os 10 países que mais cresceram e os 10 que mais encolheram (2000 → 2022)?
2. **A população idosa cresceu?**

Fonte: World Bank Open Data, indicadores `SP.POP.65UP.TO` (pessoas com 65+) e `SP.POP.65UP.TO.ZS` (% da população com 65+), baseados nas estimativas da ONU (*World Population Prospects*).

In [1]:
import json
import urllib.request
import pandas as pd
from pathlib import Path

DATA = Path('data')
ARQUIVO_IDOSOS = DATA / 'idosos_65mais_banco_mundial.csv'
ATUALIZAR = False  # True baixa os dados de novo do Banco Mundial

INDICADORES = {
    'SP.POP.65UP.TO': 'idosos_65mais',          # população com 65 anos ou mais (pessoas)
    'SP.POP.65UP.TO.ZS': 'pct_idosos_65mais',   # % da população total com 65 anos ou mais
}

## 1. Download dos dados do Banco Mundial

Os dados são salvos em `data/idosos_65mais_banco_mundial.csv`. Nas próximas execuções o notebook lê o arquivo local; para baixar de novo, mude `ATUALIZAR = True`. A API também devolve agregados (Mundo = `WLD`, regiões, faixas de renda), que usamos para o total mundial.

In [2]:
def baixar_indicador(codigo, inicio=1970, fim=2024):
    """Baixa um indicador do Banco Mundial para todos os países e agregados."""
    url = (f'https://api.worldbank.org/v2/country/all/indicator/{codigo}'
           f'?format=json&per_page=20000&date={inicio}:{fim}')
    with urllib.request.urlopen(url, timeout=60) as resposta:
        _, linhas = json.load(resposta)
    return pd.DataFrame({
        'cca3': [l['countryiso3code'] for l in linhas],
        'nome_bm': [l['country']['value'] for l in linhas],
        'ano': [int(l['date']) for l in linhas],
        INDICADORES[codigo]: [l['value'] for l in linhas],
    })

if ATUALIZAR or not ARQUIVO_IDOSOS.exists():
    partes = [baixar_indicador(c).set_index(['cca3', 'nome_bm', 'ano']) for c in INDICADORES]
    idosos = pd.concat(partes, axis=1).reset_index()
    idosos = idosos[idosos['cca3'] != '']
    idosos.to_csv(ARQUIVO_IDOSOS, index=False)
    print('Baixado do Banco Mundial e salvo em', ARQUIVO_IDOSOS)

idosos = pd.read_csv(ARQUIVO_IDOSOS)
print(idosos.shape)
idosos.head()

(14355, 5)


,cca3,nome_bm,ano,idosos_65mais,pct_idosos_65mais
0,AFE,Africa Eastern and Southern,2024,25624442.0,3.330960
1,AFE,Africa Eastern and Southern,2023,24663427.0,3.286303
2,AFE,Africa Eastern and Southern,2022,23755737.0,3.246111
3,AFE,Africa Eastern and Southern,2021,22937086.0,3.216573
4,AFE,Africa Eastern and Southern,2020,22167894.0,3.192169


## 2. Mundo: os idosos cresceram?

In [3]:
mundo = idosos[idosos['cca3'] == 'WLD'].set_index('ano').loc[[1970, 1980, 1990, 2000, 2010, 2022, 2024]]
tabela = pd.DataFrame({
    'idosos (mi)': (mundo['idosos_65mais'] / 1e6).round(0),
    '% da população': mundo['pct_idosos_65mais'].round(1),
})
tabela

,idosos (mi),% da população
ano,,
1970,194.0,5.3
1980,258.0,5.8
1990,322.0,6.1
2000,421.0,6.8
2010,531.0,7.6
2022,778.0,9.7
2024,831.0,10.2


**Sim.** O número de pessoas com 65 anos ou mais no mundo **quadruplicou** de 1970 a 2024 (de ~194 milhões para ~831 milhões), e a fatia delas na população quase dobrou (de 5,3% para 10,2%). Ou seja, os idosos cresceram muito mais rápido que a população total.

## 3. Cruzamento com a nossa base

A ligação é pelo código do país (`cca3` na nossa base = código ISO3 do Banco Mundial).

In [4]:
base = pd.read_csv(DATA / 'world_population_limpo.csv')

largo = idosos.pivot_table(index='cca3', columns='ano', values=['idosos_65mais', 'pct_idosos_65mais'])
paises = base.set_index('cca3')[['pais', 'continente', 'pop_2000', 'pop_2022']].join(
    pd.DataFrame({
        'idosos_2000': largo[('idosos_65mais', 2000)],
        'idosos_2022': largo[('idosos_65mais', 2022)],
        'pct_idosos_2000': largo[('pct_idosos_65mais', 2000)],
        'pct_idosos_2022': largo[('pct_idosos_65mais', 2022)],
    })
)

sem_dado = paises[paises[['idosos_2000', 'idosos_2022']].isna().any(axis=1)]
print(f'{len(paises) - len(sem_dado)} de {len(paises)} países/territórios com dados de idosos.')
print('Sem dados (territórios pequenos):', ', '.join(sorted(sem_dado['pais'])))
paises = paises.dropna(subset=['idosos_2000', 'idosos_2022'])

214 de 234 países/territórios com dados de idosos.
Sem dados (territórios pequenos): Anguilla, Cook Islands, Falkland Islands, French Guiana, Guadeloupe, Guernsey, Jersey, Martinique, Mayotte, Montserrat, Niue, Northern Mariana Islands, Reunion, Saint Barthelemy, Saint Pierre and Miquelon, Taiwan, Tokelau, Vatican City, Wallis and Futuna, Western Sahara


## 4. País a país (2000 → 2022)

Duas formas de medir:
- **número de idosos** (pessoas com 65+): cresce até em país que encolhe, se a população vive mais;
- **fatia de idosos** (% da população com 65+): mostra o **envelhecimento** de fato. A variação é em **pontos percentuais** (p.p.): de 10% para 15% = +5 p.p.

In [5]:
paises['cresc_pop_%'] = (paises['pop_2022'] / paises['pop_2000'] - 1) * 100
paises['cresc_idosos_%'] = (paises['idosos_2022'] / paises['idosos_2000'] - 1) * 100
paises['var_pct_idosos_pp'] = paises['pct_idosos_2022'] - paises['pct_idosos_2000']

n = len(paises)
cresceu_numero = (paises['cresc_idosos_%'] > 0).sum()
cresceu_fatia = (paises['var_pct_idosos_pp'] > 0).sum()
print(f'Número de idosos cresceu em {cresceu_numero} de {n} países ({cresceu_numero / n:.1%}).')
print(f'A fatia de idosos na população cresceu em {cresceu_fatia} de {n} países ({cresceu_fatia / n:.1%}).')

print('Onde o número de idosos caiu:')
display(paises.loc[paises['cresc_idosos_%'] <= 0, ['pais', 'idosos_2000', 'idosos_2022']])
print('Onde a fatia de idosos mais caiu:')
display(paises.nsmallest(5, 'var_pct_idosos_pp')[['pais', 'pct_idosos_2000', 'pct_idosos_2022']].round(1))

Número de idosos cresceu em 213 de 214 países (99.5%).
A fatia de idosos na população cresceu em 191 de 214 países (89.3%).
Onde o número de idosos caiu:


,pais,idosos_2000,idosos_2022
cca3,,,
LSO,Lesotho,90760.0,86711.0


Onde a fatia de idosos mais caiu:


,pais,pct_idosos_2000,pct_idosos_2022
cca3,,,
GAB,Gabon,4.9,4.0
MLI,Mali,3.2,2.4
TCD,Chad,2.8,2.1
LSO,Lesotho,4.5,3.8
MWI,Malawi,3.3,2.6


## 5. Ranking: os 10 que mais cresceram e os 10 que mais encolheram

Crescimento da população total de 2000 a 2022, só países com mais de 1 milhão de habitantes.

In [6]:
grandes = paises[paises['pop_2022'] >= 1_000_000]
colunas = ['pais', 'continente', 'cresc_pop_%', 'pct_idosos_2000', 'pct_idosos_2022',
           'var_pct_idosos_pp', 'cresc_idosos_%']

def ranking(tabela):
    t = tabela[colunas].round(1).reset_index(drop=True)
    t.index = t.index + 1
    return t

mais_cresceram = ranking(grandes.nlargest(10, 'cresc_pop_%'))
mais_encolheram = ranking(grandes.nsmallest(10, 'cresc_pop_%'))
print('Os 10 que mais cresceram, 2000-2022')
display(mais_cresceram)
print('Os 10 que mais encolheram, 2000-2022')
display(mais_encolheram)

Os 10 que mais cresceram, 2000-2022


,pais,continente,cresc_pop_%,pct_idosos_2000,pct_idosos_2022,var_pct_idosos_pp,cresc_idosos_%
1,Qatar,Asia,317.2,1.2,1.4,0.1,360.8
2,United Arab Emirates,Asia,188.2,1.3,1.6,0.4,270.9
3,Equatorial Guinea,Africa,144.5,3.3,3.6,0.2,175.0
4,Niger,Africa,125.5,2.3,2.6,0.3,149.1
5,Jordan,Asia,123.2,2.7,4.1,1.4,221.1
6,Kuwait,Asia,120.6,1.5,2.7,1.2,317.9
7,Angola,Africa,117.1,2.7,2.8,0.1,128.2
8,Chad,Africa,114.6,2.8,2.1,-0.8,57.7
9,Afghanistan,Asia,110.5,2.3,2.4,0.1,109.9
10,Bahrain,Asia,106.9,2.2,3.5,1.2,271.9


Os 10 que mais encolheram, 2000-2022


,pais,continente,cresc_pop_%,pct_idosos_2000,pct_idosos_2022,var_pct_idosos_pp,cresc_idosos_%
1,Lithuania,Europe,-23.6,13.8,19.9,6.0,16.1
2,Moldova,Europe,-23.0,10.1,15.1,5.0,29.4
3,Latvia,Europe,-22.6,15.0,21.0,6.0,11.3
4,Bosnia and Herzegovina,Europe,-22.6,11.0,21.1,10.1,48.1
5,Ukraine,Europe,-18.8,13.4,17.9,4.4,10.3
6,Bulgaria,Europe,-16.2,16.6,21.7,5.1,3.5
7,Puerto Rico,North America,-15.0,11.6,23.8,12.3,74.0
8,Armenia,Asia,-12.2,9.1,12.9,3.8,30.2
9,Georgia,Asia,-12.2,12.4,15.1,2.7,10.7
10,Croatia,Europe,-11.4,15.9,22.5,6.6,21.9


### Os dois grupos envelheceram igual?

In [7]:
resumo = pd.DataFrame({
    'grupo': ['10 que mais cresceram', '10 que mais encolheram', 'todos (1 mi+)'],
    'pct_idosos_2022_mediana': [mais_cresceram['pct_idosos_2022'].median(),
                                mais_encolheram['pct_idosos_2022'].median(),
                                grandes['pct_idosos_2022'].median()],
    'var_pct_idosos_pp_mediana': [mais_cresceram['var_pct_idosos_pp'].median(),
                                  mais_encolheram['var_pct_idosos_pp'].median(),
                                  grandes['var_pct_idosos_pp'].median()],
}).round(1)
resumo

,grupo,pct_idosos_2022_mediana,var_pct_idosos_pp_mediana
0,10 que mais cresceram,2.7,0.2
1,10 que mais encolheram,20.4,5.6
2,todos (1 mi+),6.8,2.2


**Leitura:**
- Os países que **mais encolheram** (Leste Europeu e ex-URSS) são **bem mais velhos**: cerca de 1 em cada 5 habitantes tem 65+. Quem emigra são principalmente os jovens, então a população que fica envelhece mais rápido.
- Os que **mais cresceram** são **jovens**: no Golfo, a imigração é de adultos em idade de trabalho; na África e no Afeganistão, a natalidade alta mantém a base da pirâmide larga. Mesmo assim, o número absoluto de idosos também cresceu na maioria deles.

## 6. Os países mais envelhecidos em 2022 e os que mais envelheceram desde 2000

In [8]:
display(ranking(grandes.nlargest(10, 'pct_idosos_2022')))
display(ranking(grandes.nlargest(10, 'var_pct_idosos_pp')))

,pais,continente,cresc_pop_%,pct_idosos_2000,pct_idosos_2022,var_pct_idosos_pp,cresc_idosos_%
1,Japan,Asia,-2.2,17.4,29.4,12.0,66.3
2,Italy,Europe,3.6,18.2,23.9,5.7,36.1
3,Puerto Rico,North America,-15.0,11.6,23.8,12.3,74.0
4,Portugal,Europe,-0.3,16.1,23.7,7.6,48.9
5,Finland,Europe,7.0,14.9,23.2,8.3,67.2
6,Greece,Europe,-5.9,17.0,23.2,6.2,31.8
7,Croatia,Europe,-11.4,15.9,22.5,6.6,21.9
8,Germany,Europe,2.2,16.5,22.5,6.0,38.0
9,Serbia,Europe,-9.0,16.1,22.1,6.0,21.8
10,Bulgaria,Europe,-16.2,16.6,21.7,5.1,3.5


,pais,continente,cresc_pop_%,pct_idosos_2000,pct_idosos_2022,var_pct_idosos_pp,cresc_idosos_%
1,Puerto Rico,North America,-15.0,11.6,23.8,12.3,74.0
2,Japan,Asia,-2.2,17.4,29.4,12.0,66.3
3,South Korea,Asia,10.7,7.1,17.5,10.4,171.0
4,Bosnia and Herzegovina,Europe,-22.6,11.0,21.1,10.1,48.1
5,Hong Kong,Asia,11.3,11.1,20.7,9.6,105.2
6,Albania,Europe,-10.7,7.3,15.8,8.4,70.8
7,Finland,Europe,7.0,14.9,23.2,8.3,67.2
8,Thailand,Asia,13.7,6.1,14.1,8.0,162.3
9,Portugal,Europe,-0.3,16.1,23.7,7.6,48.9
10,North Macedonia,Europe,2.7,9.8,17.0,7.3,57.9


## 7. Exportação

In [9]:
paises.reset_index().to_csv(DATA / 'crescimento_e_idosos_2000_2022.csv', index=False)
print('Salvo:', DATA / 'crescimento_e_idosos_2000_2022.csv', paises.shape)

Salvo: data\crescimento_e_idosos_2000_2022.csv (214, 11)


## Conclusão

- **A população idosa cresceu**: no mundo, o número de pessoas com 65+ quase dobrou de 2000 a 2022 (421 → 778 milhões). País a país, o número de idosos aumentou em 213 de 214 (a exceção é o Lesoto, afetado pela epidemia de HIV/aids) e a fatia de idosos aumentou em 191 de 214 (89%). As quedas de fatia ocorrem em países africanos de natalidade muito alta (Gabão, Mali, Chade), onde a população jovem cresce ainda mais rápido que a idosa.
- Crescimento populacional e envelhecimento andam em sentidos opostos: quem mais encolhe é quem mais envelhece.
- Limitação: os dados de idade vêm de uma fonte externa (Banco Mundial/ONU), não da base original do projeto; alguns territórios pequenos ficam de fora por falta de dados.